With slight modifications, the code is taken from the tutorial by Sterbak (2018) available here: https://www.depends-on-the-definition.com/named-entity-recognition-with-bert/

upload data set to colab

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving PV_DATASET_COLAB.csv to PV_DATASET_COLAB.csv


import libraries
import random seed for reproducibility of results

In [ ]:
import numpy as np
import torch
torch.manual_seed(200)

some data gets lost when read from pandas

import three columns of data ('id', 'token', 'tag') into three lists to solve this problem

In [ ]:
list1 = list()
list2 = list()
list3 = list()

In [ ]:
fil = open('dataset_fine_tuning.csv', 'r', encoding='utf8')
lines = fil.readlines()
for line in lines:
  line = line.split()
  list1.append(line[0])
  list2.append(line[1])
  list3.append(line[2])

convert three lists into pandas dataframe and check the result

In [ ]:
import pandas as pd
data = pd.DataFrame(list(zip(list1, list2, list3)),
                columns =['id', 'token', 'tag'])
print(data.head(10))

  id    token tag
0  0       Is  NO
1  0  Sadness  NO
2  0        a  NO
3  0  Disease  NO
4  0        ?  NO
5  1      NEW  NO
6  1     YORK  NO
7  1        –  NO
8  1  Sadness  NO
9  1       is  NO


retrieve sentences with sentence id and tags

In [ ]:
class SentenceGetter(object):

  def __init__(self, data):
    self.n_sent = 1
    self.data = data
    agg_func = lambda s: [(tk, tg) for tk, tg in zip(s['token'].values.tolist(),
                                                     s['tag'].values.tolist())]
    self.grouped = self.data.groupby('id').apply(agg_func)
    self.sentences = [s for s in self.grouped]

  def get_next(self):
    try:
      s = self.grouped['{}'.format(self.n_sent)]
      self.n_sent += 1
      return s
    except:
      return None



In [ ]:
getter = SentenceGetter(data)

every sentence is represented as a list of tokens

In [ ]:
sentences = [[word[0] for word in sentence] for sentence in getter.sentences]

represent token labels for every sentence as a list of labels

In [ ]:
labels = [[s[1] for s in sentence] for sentence in getter.sentences]

add PAD for further padding as third label; map all labels to indices

In [ ]:
tag_values = list(set(data['tag'].values))
tag_values.append('PAD')
tag2idx = {t: i for i, t in enumerate(tag_values)}

In [ ]:
tag2idx

{'NO': 0, 'PAD': 2, 'YES': 1}

In [ ]:
!pip install transformers

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertConfig

In [ ]:
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

torch.__version__


'1.8.1+cu101'

check GPU availability

In [ ]:
torch.cuda.get_device_name(0)

'Tesla T4'

introduce maximum sequence length and batch size

In [ ]:
MAX_LEN = 108
bs = 64

In [ ]:
device = torch.device("cuda")
n_gpu = torch.cuda.device_count()

load Bert tokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case = True)

tokenize sentences; add labels for all subwords after wordpiece tokenization


In [ ]:
def tokenize_and_preserve_labels(sentence, text_labels):
    tokenized_sentence = []
    labels = []

    for word, label in zip(sentence, text_labels):

        # Tokenize the word and count # of subwords the word is broken into
        tokenized_word = tokenizer.tokenize(word)
        n_subwords = len(tokenized_word)

        # Add the tokenized word to the final tokenized word list
        tokenized_sentence.extend(tokenized_word)

        # Add the same label to the new list of labels `n_subwords` times
        labels.extend([label] * n_subwords)

    return tokenized_sentence, labels

In [ ]:
tokenized_texts_and_labels = [
    tokenize_and_preserve_labels(sent, labs)
    for sent, labs in zip(sentences, labels)
]

In [ ]:
tokenized_texts = [token_label_pair[0] for token_label_pair in tokenized_texts_and_labels]
labels = [token_label_pair[1] for token_label_pair in tokenized_texts_and_labels]

map tokens to vocabulary indices and pad all sentences to MAX_LEN with PAD;

In [ ]:
input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", value=0.0,
                          truncating="post", padding="post")

In [ ]:
tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in labels],
                     maxlen=MAX_LEN, value=tag2idx["PAD"], padding="post",
                     dtype="long", truncating="post")

introduce attention masks to ignore PADs in training

In [ ]:
attention_masks = [[float(i != 0.0) for i in ii] for ii in input_ids]

split data set 80:20 for training and validation

In [ ]:
tr_inputs, val_inputs, tr_tags, val_tags = train_test_split(input_ids, tags,
                                                            random_state=200, test_size=0.2)
tr_masks, val_masks, _, _ = train_test_split(attention_masks, input_ids,
                                             random_state=200, test_size=0.2)

convert data to torch tensors

In [ ]:
tr_inputs = torch.tensor(tr_inputs)
val_inputs = torch.tensor(val_inputs)
tr_tags = torch.tensor(tr_tags)
val_tags = torch.tensor(val_tags)
tr_masks = torch.tensor(tr_masks)
val_masks = torch.tensor(val_masks)

define data loaders; introduce random shuffling; prepare to pass data as sequence during validation

In [ ]:
train_data = TensorDataset(tr_inputs, tr_masks, tr_tags)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=bs)

valid_data = TensorDataset(val_inputs, val_masks, val_tags)
valid_sampler = SequentialSampler(valid_data)
valid_dataloader = DataLoader(valid_data, sampler=valid_sampler, batch_size=bs)

import Bert model and optimizer

In [ ]:
import transformers
from transformers import BertForTokenClassification, AdamW

transformers.__version__

'4.6.1'

In [ ]:
len(tag2idx)

3

initialize model from pretrained

In [ ]:
model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(tag2idx),
    output_attentions = False,
    output_hidden_states = False
)

pass model to GPU

In [ ]:
model.cuda();

perform full fine-tuning

In [ ]:
FULL_FINETUNING = True
if FULL_FINETUNING:
    param_optimizer = list(model.named_parameters())
    no_decay = ['bias', 'gamma', 'beta']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0}
    ]
else:
    param_optimizer = list(model.classifier.named_parameters())
    optimizer_grouped_parameters = [{"params": [p for n, p in param_optimizer]}]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=1e-4,
    eps=1e-8
)

import scheduler for to gradually reduce learning rate during the training

In [ ]:
from transformers import get_linear_schedule_with_warmup

epochs = 3
max_grad_norm = 1.0

# Total number of training steps is number of batches * number of epochs.
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

install seqeval and install necessary metrics

In [ ]:
!pip install 'seqeval == 0.0.12'

In [ ]:
from seqeval.metrics import f1_score, accuracy_score

import trange to show progress throughout looping

In [ ]:
from tqdm import trange

perform training and validation and display evaluation results

In [ ]:
## Store the average loss after each epoch so we can plot them.
loss_values, validation_loss_values = [], []

for _ in trange(epochs, desc="Epoch"):
    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set.

    # Put the model into training mode.
    model.train()
    # Reset the total loss for this epoch.
    total_loss = 0

    # Training loop
    for step, batch in enumerate(train_dataloader):
        # add batch to gpu
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch
        # Always clear any previously calculated gradients before performing a backward pass.
        model.zero_grad()
        # forward pass
        # This will return the loss (rather than the model output)
        # because we have provided the `labels`.
        outputs = model(b_input_ids, token_type_ids=None,
                        attention_mask=b_input_mask, labels=b_labels)
        # get the loss
        loss = outputs[0]
        # Perform a backward pass to calculate the gradients.
        loss.backward()
        # track train loss
        total_loss += loss.item()
        # Clip the norm of the gradient
        # This is to help prevent the "exploding gradients" problem.
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
        # update parameters
        optimizer.step()
        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over the training data.
    avg_train_loss = total_loss / len(train_dataloader)
    print("Average train loss: {}".format(avg_train_loss))

    # Store the loss value for plotting the learning curve.
    loss_values.append(avg_train_loss)


    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.

    # Put the model into evaluation mode
    model.eval()
    # Reset the validation loss for this epoch.
    eval_loss, eval_accuracy = 0, 0
    nb_eval_steps, nb_eval_examples = 0, 0
    predictions , true_labels = [], []
    for batch in valid_dataloader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch

        # Telling the model not to compute or store gradients,
        # saving memory and speeding up validation
        with torch.no_grad():
            # Forward pass, calculate logit predictions.
            # This will return the logits rather than the loss because we have not provided labels.
            outputs = model(b_input_ids, token_type_ids=None,
                            attention_mask=b_input_mask, labels=b_labels)
        # Move logits and labels to CPU
        logits = outputs[1].detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the accuracy for this batch of test sentences.
        eval_loss += outputs[0].mean().item()
        predictions.extend([list(p) for p in np.argmax(logits, axis=2)])
        true_labels.extend(label_ids)

    eval_loss = eval_loss / len(valid_dataloader)
    validation_loss_values.append(eval_loss)
    print("Validation loss: {}".format(eval_loss))
    pred_tags = [tag_values[p_i] for p, l in zip(predictions, true_labels)
                                 for p_i, l_i in zip(p, l) if tag_values[l_i] != "PAD"]
    valid_tags = [tag_values[l_i] for l in true_labels
                                  for l_i in l if tag_values[l_i] != "PAD"]
    print("Validation Accuracy: {}".format(accuracy_score(pred_tags, valid_tags)))
    print("Validation F1-Score: {}".format(f1_score(pred_tags, valid_tags)))
    print()


visualize learning curve

In [ ]:
from mlxtend.plotting import plot_confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
from sklearn.metrics import plot_confusion_matrix

# Use plot styling from seaborn.
sns.set(style='darkgrid')

# Increase the plot size and font size.
sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)

# Plot the learning curve.
plt.plot(loss_values, 'b-o', label="training loss")
plt.plot(validation_loss_values, 'r-o', label="validation loss")

# Label the plot.
plt.title("Learning curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

display confusion matrix

In [ ]:
from sklearn import metrics
cm=metrics.confusion_matrix(pred_tags, valid_tags)
print(cm)

test the model on additional sentences to see how it works

In [ ]:
test_sentence = """
"""


In [ ]:
tokenized_sentence = tokenizer.encode(test_sentence)
input_ids = torch.tensor([tokenized_sentence]).cuda()

In [ ]:
with torch.no_grad():
    output = model(input_ids)
label_indices = np.argmax(output[0].to('cpu').numpy(), axis=2)

In [ ]:
# join bpe split tokens
tokens = tokenizer.convert_ids_to_tokens(input_ids.to('cpu').numpy()[0])
new_tokens, new_labels = [], []
for token, label_idx in zip(tokens, label_indices[0]):
    if token.startswith("##"):
        new_tokens[-1] = new_tokens[-1] + token[2:]
    else:
        new_labels.append(tag_values[label_idx])
        new_tokens.append(token)

In [ ]:
for token, label in zip(new_tokens, new_labels):
    print("{}\t{}".format(label, token))

save model to google drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import torch
import transformers

In [ ]:
model_save_name = 'FINAL_BEST.pt'
path = F"/content/gdrive/My Drive/{model_save_name}"
torch.save(model, path)
